In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import uuid

In [0]:
#Generate pipeline run id
pipeline_run_id = str(uuid.uuid4())

In [0]:
orders_df = spark.table("ecom_medallion_pipeline.bronze.orders_bronze")
products_df = spark.table("ecom_medallion_pipeline.bronze.products_bronze")

In [0]:
def blank_as_null(column_name):
    return F.when(F.trim(F.col(column_name)) == "", None).otherwise(F.col(column_name))

In [0]:
orders_prepared_df = (
    orders_df
    .withColumn("order_id", blank_as_null("order_id"))
    .withColumn("customer_id", blank_as_null("customer_id"))
    .withColumn("product_id", blank_as_null("product_id"))
    .withColumn("order_status_std", F.lower(F.trim(F.col("order_status"))))
    .withColumn("quantity_num", F.col("quantity").cast("int"))
    .withColumn("unit_price_num", F.col("unit_price").cast("double"))
    .withColumn(
        "order_timestamp",
        F.coalesce(
            F.try_to_timestamp(F.col("order_date"), F.lit("yyyy-MM-dd HH:mm:ss")),
            F.try_to_timestamp(F.col("order_date"), F.lit("yyyy-MM-dd'T'HH:mm:ss"))
        )
    )
)

In [0]:
order_dup_window = Window.partitionBy("order_id").orderBy(
    F.col("load_date").asc(),
    F.col("source_file_name").asc(),
    F.col("ingestion_timestamp").asc()
)

orders_with_dup_flag_df = (orders_prepared_df.withColumn("dup_rank", F.row_number().over(order_dup_window)))


In [0]:
orders_validated_df = (
    orders_with_dup_flag_df
    .withColumn(
        "rejection_reason",
        F.when(F.col("order_id").isNull(), F.lit("NULL_ORDER_ID"))
         .when(F.col("customer_id").isNull(), F.lit("NULL_CUSTOMER_ID"))
         .when(F.col("product_id").isNull(), F.lit("NULL_PRODUCT_ID"))
         .when(F.col("order_timestamp").isNull(), F.lit("INVALID_ORDER_DATE"))
         .when(F.col("quantity_num").isNull(), F.lit("NULL_OR_INVALID_QUANTITY"))
         .when(F.col("quantity_num") <= 0, F.lit("INVALID_QUANTITY"))
         .when(F.col("unit_price_num").isNull(), F.lit("NULL_OR_INVALID_UNIT_PRICE"))
         .when(F.col("unit_price_num") <= 0, F.lit("INVALID_UNIT_PRICE"))
         .when(~F.col("order_status_std").isin("completed", "pending", "cancelled"), F.lit("INVALID_ORDER_STATUS"))
         .when((F.col("order_id").isNotNull()) & (F.col("dup_rank") > 1), F.lit("DUPLICATE_ORDER_ID"))
         .otherwise(F.lit(None))
    )
)

In [0]:
# Separate rejected and valid orders

rejected_orders_df = (
    orders_validated_df
    .filter(F.col("rejection_reason").isNotNull())
    .select(
        F.to_json(
            F.struct(
                F.col("order_id"),
                F.col("customer_id"),
                F.col("product_id"),
                F.col("order_date"),
                F.col("quantity"),
                F.col("unit_price"),
                F.col("order_status")
            )
        ).alias("raw_record"),
        F.col("rejection_reason"),
        F.col("source_file_name"),
        F.col("load_date"),
        F.lit(pipeline_run_id).alias("pipeline_run_id")
    )
)

valid_orders_df = (
    orders_validated_df
    .filter(F.col("rejection_reason").isNull())
)

In [0]:
products_prepared_df = (
    products_df
    .withColumn("product_id", blank_as_null("product_id"))
    .withColumn("product_name", blank_as_null("product_name"))
    .withColumn("category", blank_as_null("category"))
    .withColumn("brand", blank_as_null("brand"))
)

products_valid_df = (
    products_prepared_df
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("product_name").isNotNull())
    .filter(F.col("category").isNotNull())
    .filter(F.col("brand").isNotNull())
)

In [0]:
product_dup_window = Window.partitionBy("product_id").orderBy(
    F.col("load_date").asc(),
    F.col("source_file_name").asc(),
    F.col("ingestion_timestamp").asc()
)

products_dedup_df = (
    products_valid_df
    .withColumn("prod_rank", F.row_number().over(product_dup_window))
    .filter(F.col("prod_rank") == 1)
    .select(
        F.col("product_id").alias("p_product_id"),
        F.col("product_name"),
        F.col("category"),
        F.col("brand")
    )
)

In [0]:
orders_curated_df = (
    valid_orders_df.alias("o")
    .join(
        products_dedup_df.alias("p"),
        F.col("o.product_id") == F.col("p.p_product_id"),
        "left"
    )
    .select(
        F.col("o.order_id").alias("order_id"),
        F.col("o.customer_id").alias("customer_id"),
        F.col("o.product_id").alias("product_id"),
        F.coalesce(F.col("p.product_name"), F.lit("UNKNOWN")).alias("product_name"),
        F.coalesce(F.col("p.category"), F.lit("UNKNOWN")).alias("category"),
        F.coalesce(F.col("p.brand"), F.lit("UNKNOWN")).alias("brand"),
        F.col("o.order_timestamp").alias("order_timestamp"),
        F.to_date(F.col("o.order_timestamp")).alias("order_date"),
        F.col("o.quantity_num").alias("quantity"),
        F.col("o.unit_price_num").alias("unit_price"),
        (F.col("o.quantity_num") * F.col("o.unit_price_num")).cast("double").alias("order_amount"),
        F.col("o.order_status_std").alias("order_status"),
        F.col("o.load_date").alias("load_date"),
        F.lit(pipeline_run_id).alias("pipeline_run_id")
    )
)

#### Write Silver Curated Orders table and Write Silver Rejected Orders table

In [0]:
orders_curated_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecom_medallion_pipeline.silver.orders_curated")

In [0]:
rejected_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecom_medallion_pipeline.silver.rejected_orders")


#### Data Validation

In [0]:
print("Pipeline Run ID:", pipeline_run_id)

print("Curated Orders Count:")
spark.sql("SELECT COUNT(*) AS total_rows FROM ecom_medallion_pipeline.silver.orders_curated").show()

print("Rejected Orders Count:")
spark.sql("SELECT COUNT(*) AS total_rows FROM ecom_medallion_pipeline.silver.rejected_orders").show()

print("Rejection Summary:")
spark.sql("""
SELECT rejection_reason, COUNT(*) AS cnt
FROM ecom_medallion_pipeline.silver.rejected_orders
GROUP BY rejection_reason
ORDER BY cnt DESC
""").show()

print("Curated Orders Preview:")
display(spark.table("ecom_medallion_pipeline.silver.orders_curated"))

print("Rejected Orders Preview:")
display(spark.table("ecom_medallion_pipeline.silver.rejected_orders"))

Pipeline Run ID: becdc7b7-788f-44df-8208-f1f0e57da34d
Curated Orders Count:
+----------+
|total_rows|
+----------+
|        33|
+----------+

Rejected Orders Count:
+----------+
|total_rows|
+----------+
|        20|
+----------+

Rejection Summary:
+--------------------+---+
|    rejection_reason|cnt|
+--------------------+---+
|  INVALID_ORDER_DATE|  4|
|  INVALID_UNIT_PRICE|  4|
|    INVALID_QUANTITY|  4|
|INVALID_ORDER_STATUS|  2|
|     NULL_PRODUCT_ID|  2|
|    NULL_CUSTOMER_ID|  2|
|       NULL_ORDER_ID|  1|
|  DUPLICATE_ORDER_ID|  1|
+--------------------+---+

Curated Orders Preview:


order_id,customer_id,product_id,product_name,category,brand,order_timestamp,order_date,quantity,unit_price,order_amount,order_status,load_date,pipeline_run_id
O1001,C101,P100,Wireless Mouse,Accessories,LogiTech,2026-03-20T09:15:00.000Z,2026-03-20,2,599.0,1198.0,completed,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
O1002,C102,P101,Mechanical Keyboard,Accessories,KeyPro,2026-03-20T09:32:00.000Z,2026-03-20,1,1299.0,1299.0,completed,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
O1003,C103,P102,USB Hub,Accessories,Portify,2026-03-20T09:45:00.000Z,2026-03-20,3,199.0,597.0,pending,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
O1004,C104,P103,Webcam HD,Electronics,VisionTech,2026-03-20T10:10:00.000Z,2026-03-20,1,799.0,799.0,completed,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
O1005,C105,P104,27 Inch Monitor,Electronics,ViewMax,2026-03-20T10:25:00.000Z,2026-03-20,2,1499.0,2998.0,completed,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
O1006,C106,P105,Laptop Stand,Accessories,ErgoLift,2026-03-20T10:40:00.000Z,2026-03-20,1,2499.0,2499.0,cancelled,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
O1007,C107,P106,Mouse Pad,Accessories,DeskPro,2026-03-20T11:05:00.000Z,2026-03-20,4,99.0,396.0,pending,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
O1008,C108,P107,Noise Cancelling Headphones,Audio,SoundMax,2026-03-20T11:20:00.000Z,2026-03-20,1,3999.0,3999.0,completed,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
O1009,C109,P108,Portable SSD 1TB,Storage,DataFlash,2026-03-20T11:48:00.000Z,2026-03-20,2,899.0,1798.0,completed,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
O1010,C110,P109,USB-C Charger,Power,ChargeFast,2026-03-20T12:05:00.000Z,2026-03-20,1,699.0,699.0,pending,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d


Rejected Orders Preview:


raw_record,rejection_reason,source_file_name,load_date,pipeline_run_id
"{""customer_id"":""C116"",""product_id"":""P104"",""order_date"":""2026-03-20 13:40:00"",""quantity"":1,""unit_price"":1499.0,""order_status"":""completed""}",NULL_ORDER_ID,dbfs:/Volumes/ecom_medallion_pipeline/data_layer/source_1/2026-04-30/orders_001.json,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
"{""order_id"":""O1014"",""customer_id"":""C114"",""product_id"":""P102"",""order_date"":""2026-03-20 13:06:00"",""quantity"":0,""unit_price"":199.0,""order_status"":""completed""}",INVALID_QUANTITY,dbfs:/Volumes/ecom_medallion_pipeline/data_layer/source_1/2026-04-30/orders_001.json,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
"{""order_id"":""O1015"",""customer_id"":""C115"",""product_id"":""P103"",""order_date"":""2026-03-20 13:20:00"",""quantity"":-2,""unit_price"":799.0,""order_status"":""completed""}",INVALID_QUANTITY,dbfs:/Volumes/ecom_medallion_pipeline/data_layer/source_1/2026-04-30/orders_001.json,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
"{""order_id"":""O1017"",""product_id"":""P105"",""order_date"":""2026-03-20 13:55:00"",""quantity"":1,""unit_price"":2499.0,""order_status"":""pending""}",NULL_CUSTOMER_ID,dbfs:/Volumes/ecom_medallion_pipeline/data_layer/source_1/2026-04-30/orders_001.json,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
"{""order_id"":""O1018"",""customer_id"":""C118"",""order_date"":""2026-03-20 14:10:00"",""quantity"":1,""unit_price"":349.0,""order_status"":""completed""}",NULL_PRODUCT_ID,dbfs:/Volumes/ecom_medallion_pipeline/data_layer/source_1/2026-04-30/orders_001.json,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
"{""order_id"":""O1019"",""customer_id"":""C119"",""product_id"":""P106"",""order_date"":""2026-03-20 14:25:00"",""quantity"":2,""unit_price"":0.0,""order_status"":""completed""}",INVALID_UNIT_PRICE,dbfs:/Volumes/ecom_medallion_pipeline/data_layer/source_1/2026-04-30/orders_001.json,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
"{""order_id"":""O1020"",""customer_id"":""C120"",""product_id"":""P107"",""order_date"":""2026-03-20 14:40:00"",""quantity"":1,""unit_price"":-3999.0,""order_status"":""completed""}",INVALID_UNIT_PRICE,dbfs:/Volumes/ecom_medallion_pipeline/data_layer/source_1/2026-04-30/orders_001.json,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
"{""order_id"":""O1021"",""customer_id"":""C121"",""product_id"":""P108"",""order_date"":""2026/03/20 14:55:00"",""quantity"":1,""unit_price"":899.0,""order_status"":""completed""}",INVALID_ORDER_DATE,dbfs:/Volumes/ecom_medallion_pipeline/data_layer/source_1/2026-04-30/orders_001.json,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
"{""order_id"":""O1022"",""customer_id"":""C122"",""product_id"":""P109"",""order_date"":""20-03-2026 15:10:00"",""quantity"":2,""unit_price"":699.0,""order_status"":""pending""}",INVALID_ORDER_DATE,dbfs:/Volumes/ecom_medallion_pipeline/data_layer/source_1/2026-04-30/orders_001.json,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
"{""order_id"":""O1023"",""customer_id"":""C123"",""product_id"":""P100"",""order_date"":""2026-13-20 15:25:00"",""quantity"":1,""unit_price"":599.0,""order_status"":""completed""}",INVALID_ORDER_DATE,dbfs:/Volumes/ecom_medallion_pipeline/data_layer/source_1/2026-04-30/orders_001.json,2026-04-30,becdc7b7-788f-44df-8208-f1f0e57da34d
